# Week 9 — Put the model call inside a function

**Research task:** Ask one simulated actor to choose its next action after observing another actor.

**Python introduced:** `def`, explicit parameters, local variables, `return` and calling one function with changed arguments.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session09/session09_abm_updates.ipynb)

Colab supports OpenRouter only; use local Jupyter for dual-route work.

In [ ]:
# Colab setup for the OpenRouter route.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo=SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists(): setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)],check=True)
    setup_os.chdir(setup_repo/'workbook'/'session09')
print('Working folder:',SetupPath.cwd())

## Load the course settings and SDKs

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Choose a normal route and define the action schema

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
ROUTE = "ollama"
action_schema = {
    "type":"object",
    "properties":{"action":{"type":"string"},"reason":{"type":"string"}},
    "required":["action","reason"],"additionalProperties":False,
}

## Define the first student-edited model function

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
def choose_action(state, observation, route, temperature):
    prompt = (
        "Choose one next action for this simulated actor. State: " + json.dumps(state) +
        " Observation: " + observation + " Return action and reason as JSON."
    )
    messages = [{"role":"user","content":prompt}]

    if route == "openrouter":
        with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
            response = client.chat.send(
                model=HOSTED_MODEL, messages=messages, temperature=temperature,
                response_format={"type":"json_schema","json_schema":{
                    "name":"agent_action","strict":True,"schema":action_schema,
                }},
            )
        raw_output = response.choices[0].message.content
    else:
        response = ollama.chat(
            model=LOCAL_MODEL, messages=messages, format=action_schema,
            options={"temperature":temperature},
        )
        raw_output = response.message.content
    print("Raw function output:", raw_output)
    return json.loads(raw_output)

## Call the function and inspect its returned dictionary

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
state = {"speaker":"A","prior_action":"silent","goal":"be heard without conflict"}
observation = "Speaker B expresses disagreement in a calm tone."
result_1 = choose_action(state, observation, ROUTE, 0)
print("Returned value:", result_1)
print("Returned type:", type(result_1))

## Change one argument and call the same function again

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
# ONE CHANGE: the observation changes; state, route and temperature stay fixed.
changed_observation = "Speaker B interrupts and raises their voice."
result_2 = choose_action(state, changed_observation, ROUTE, 0)
print("First action:", result_1["action"])
print("Second action:", result_2["action"])

## Methodological check

A changed action shows sensitivity to the supplied observation. It does not establish that the update rule represents how real actors perceive or respond.
## Recording

Explain what enters each of the four function parameters, what remains local inside the function and what the returned dictionary contains. Compare the two calls without treating the model's reason as observed motivation.